# Notebook 6: How to Run Statistics
### Contrast, ANOVA, Clustering, and Neurobehavioral Correlations

In this notebook, we demonstrate how to run statistical tests on hyperscanning measures. We cover:
- **Univariate tests (contrasts):** Using permutation t-tests (via MNE and via HyPyP's `statsCond`).
- **Cluster-based permutation testing:** To assess significance over sensor space (using `statscondCluster`).
- **Group comparisons for connectivity:** For both intra- and inter-brain connectivity.
- **Neurobehavioral correlations:** Correlating PSD or connectivity with a behavioral measure.
  
We use functions from `hypyp.prep`, `hypyp.analyses`, `hypyp.stats`, and `hypyp.viz`.

In [ ]:
import os
import pickle
import numpy as np
import mne
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.stats as sp_stats
from collections import OrderedDict

# Import HyPyP modules
from hypyp import prep, analyses, stats, viz

# For reproducibility
np.random.seed(42)

print("Libraries and HyPyP modules imported successfully.")

## 1. Data Loading from Preprocessed Files

Instead of reloading the raw epochs and performing ICA/AutoReject preprocessing again, we'll load the preprocessed data that was saved in the first notebook. This allows us to skip the time-consuming preprocessing steps and continue directly with our analysis.

In [ ]:
try:

    filename = './data/generated/hyperscanning_data.pkl'
    #filename = './data/generated/hyperscanning_recorded_data.pkl'
    
    with open(filename, 'rb') as f:
        saved_data = pickle.load(f)
    
    # Extract the cleaned epochs from the saved data
    preproc_S1 = saved_data['epo1_clean']
    preproc_S2 = saved_data['epo2_clean']
    
    print("Successfully loaded preprocessed data from pickle file.")
    print("Participant 1:", preproc_S1)
    print("Participant 2:", preproc_S2)
    
except FileNotFoundError:
    print("Pickle file not found. Falling back to loading and preprocessing .fif files.")

## 2. PSD Computation and Univariate Statistical Testing

We compute the Power Spectral Density (PSD) in the Alpha-Low band for each participant using `analyses.pow`. Then we combine the PSD data and run a permutation t-test. Finally, we also compute the statistical conditions using HyPyP's `statsCond` function.

In [ ]:
# Compute PSD for participant 1 (Alpha-Low: 7.5-11 Hz)
psd1 = analyses.pow(preproc_S1, fmin=7.5, fmax=11, n_fft=1000, n_per_seg=1000, epochs_average=True)
# Compute PSD for participant 2 (Alpha-Low)
psd2 = analyses.pow(preproc_S2, fmin=7.5, fmax=11, n_fft=1000, n_per_seg=1000, epochs_average=True)

# Combine PSD data into a single array: shape (2, n_channels, n_freq)
data_psd = np.array([psd1.psd, psd2.psd])
print("PSD analysis completed. Data shape:", data_psd.shape)

### Univariate Permutation t-test on PSD

We now perform a permutation t-test to see if the mean PSD (across channels and frequencies) significantly deviates from 0.  
The `statsCond` function expects a 3D array (n_samples, n_tests, n_freq). For demonstration, we simulate multiple “samples” by replicating the PSD data and then add a new frequency dimension.

In [ ]:
# Simulate multiple samples (e.g., across subjects or trials)
n_samples = 20
noise_level = 1e-6
psd = data_psd

# Get the actual number of channels
n_channels = psd.shape[1]
print(f"Number of channels: {n_channels}")

# Create a temporary copy of the epochs object with only EEG channels
epochs_eeg = preproc_S1.copy().pick_types(eeg=True, exclude=[])
n_eeg_channels = len(epochs_eeg.ch_names)
print(f"Number of EEG channels: {n_eeg_channels}")

# Make sure we're only using EEG channels for our statistics
if n_channels > n_eeg_channels:
    print(f"Warning: Data contains {n_channels} channels but only {n_eeg_channels} are EEG. Using only EEG channels.")
    # If needed, select only EEG channels from psd
    # This step depends on how your channel order is structured - adjust if needed
    eeg_psd = psd[:, :n_eeg_channels, :]
else:
    eeg_psd = psd

# Create a sample data array using only EEG channels
data_sample = np.array([
    eeg_psd[0, :, 0] + 
    noise_level * np.random.randn(n_eeg_channels) + 
    noise_level * np.concatenate([np.zeros((n_eeg_channels-10,)), np.ones((10,))]) 
    for _ in range(n_samples)
])

# Add a frequency dimension (here, 1 frequency bin)
data_sample = data_sample[..., np.newaxis]  # shape: (n_samples, n_tests, 1)
print("Data for statsCond shape:", data_sample.shape)

# Run permutation t-test using HyPyP's statsCond function, with EEG-only epochs
T_obs, p_values, H0, adj_p, T_obs_plot = stats.statsCond(data_sample, epochs_eeg, n_permutations=5000, alpha=0.05)
print("Permutation t-test completed.")
print("T_obs shape:", T_obs.shape)

# Plot sensor-level T-values (all sensors) using the averaged T-values
viz.plot_significant_sensors(T_obs_plot=T_obs, epochs=epochs_eeg, significant=T_obs_plot)
print("Sensor-level T-values plotted (all sensors).")

## 3. Cluster-Based Permutation Testing

We now simulate two conditions by adding small noise to the PSD data and then run cluster-based permutation tests using `statscondCluster`.
We first compute an a priori connectivity matrix (using `con_matrix`) and convert it to a SciPy sparse matrix.

In [ ]:
psd[0, :, 0]

In [ ]:
# Ensure we're working with only EEG channels
epochs_eeg = preproc_S1.copy().pick_types(eeg=True, exclude=[])
n_eeg_channels = len(epochs_eeg.ch_names)
print(f"Number of EEG channels: {n_eeg_channels}")

# Check if we need to adjust our PSD data
if psd.shape[1] > n_eeg_channels:
    print(f"Adjusting PSD data to use only EEG channels ({n_eeg_channels})")
    eeg_psd = psd[:, :n_eeg_channels, :]
else:
    eeg_psd = psd

# Simulate two conditions by adding small noise to the PSD data
data_condition1 = np.array([
    eeg_psd[0, :, 0] + noise_level * np.random.randn(n_eeg_channels) 
    for _ in range(n_samples)
])

data_condition2 = np.array([
    eeg_psd[0, :, 0] + noise_level * np.random.randn(n_eeg_channels) + 
    noise_level * np.concatenate([np.zeros((n_eeg_channels-10,)), np.ones((10,))]) 
    for _ in range(n_samples)
])

# Create a list of data arrays (one per condition)
data_list = [data_condition1, data_condition2]

# Create connectivity matrix for a priori sensor connectivity using EEG-only epochs
con_matrixTuple = stats.con_matrix(epochs_eeg, freqs_mean=[psd1.freq_list[0]], draw=True)
ch_con_freq = con_matrixTuple.ch_con_freq

# Convert ch_con_freq to a SciPy sparse matrix (choose bsr or csr; here we use csr)
ch_con_freq_sparse = sp.csr_matrix(ch_con_freq)

# Run cluster-level permutation test (e.g., 5000 permutations, alpha=0.05)
F_obs, clusters, cluster_p_values, H0_cluster, F_obs_plot = stats.statscondCluster(
    data_list,
    freqs_mean=[psd1.freq_list[0]],
    ch_con_freq=ch_con_freq_sparse,
    tail=0,
    n_permutations=5000,
    alpha=0.05
)

print("Cluster-based permutation test for PSD completed.")
print("F_obs shape:", F_obs.shape)
print("Number of clusters found:", len(cluster_p_values))

plt.figure()
viz.plot_significant_sensors(T_obs_plot=F_obs, epochs=epochs_eeg, significant=F_obs_plot)

## 4. Intra- and Inter-Brain Connectivity Cluster Tests

For further demonstration, we create fake groups for intra-brain and inter-brain connectivity tests.
We use the connectivity results (here simulated as `result_intra` and `values`) from previous analyses.

In [ ]:
# Here, for demonstration, we simulate connectivity data.
# Let's assume result_intra is a tuple or list containing connectivity matrices for two conditions (intra-brain)
# For demonstration, we simulate two random connectivity matrices of shape (n_channels, n_channels)
n_channels = len(preproc_S1.info['ch_names'])
result_intra = [np.random.rand(n_channels, n_channels), np.random.rand(n_channels, n_channels)]

mask = np.zeros(result_intra[0].shape)
mask[26:,:][:, 22:] = 1

# Create fake groups for intra-brain connectivity analysis by replicating and adding slight noise
Alpha_Low = [
    np.array([result_intra[0] + np.random.normal(0, noise_level, result_intra[0].shape) + noise_level * mask,
              result_intra[0] + np.random.normal(0, noise_level, result_intra[0].shape) + noise_level * mask,
              result_intra[0] + np.random.normal(0, noise_level, result_intra[0].shape) + noise_level * mask,
              result_intra[0] + np.random.normal(0, noise_level, result_intra[0].shape) + noise_level * mask,
              result_intra[0] + np.random.normal(0, noise_level, result_intra[0].shape) + noise_level * mask]),
    np.array([result_intra[1] + np.random.normal(0, noise_level, result_intra[1].shape),
              result_intra[1] + np.random.normal(0, noise_level, result_intra[1].shape),
              result_intra[1] + np.random.normal(0, noise_level, result_intra[1].shape),
              result_intra[1] + np.random.normal(0, noise_level, result_intra[1].shape),
              result_intra[1] + np.random.normal(0, noise_level, result_intra[1].shape)])
]

# Create fake groups for inter-brain connectivity analysis.
# For demonstration, assume "values" is a connectivity matrix (e.g., from inter-brain analysis) of shape (n_channels, n_channels)
values = np.random.rand(n_channels, n_channels)
data_inter = [
    np.array([values, 
              values, 
              values, 
              values, 
              values]),
    np.array([values + np.random.normal(0, noise_level, result_intra[0].shape),
              values + np.random.normal(0, noise_level, result_intra[0].shape),
              values + np.random.normal(0, noise_level, result_intra[0].shape),
              values + np.random.normal(0, noise_level, result_intra[0].shape),
              values + np.random.normal(0, noise_level, result_intra[0].shape)])
]

print("Fake groups for connectivity analysis created. Shapes:",
      [arr.shape for arr in data_inter])

### Cluster Tests for Intra-Brain Connectivity

We now run a cluster-based permutation test on the fake intra-brain connectivity data.

In [ ]:
# Create connectivity matrix for intra-brain using participant 1's sensor layout
con_matrixTuple_intra = stats.con_matrix(preproc_S1, freqs_mean=np.arange(7.5, 11), draw=False)
ch_con_intra = con_matrixTuple_intra.ch_con

# Convert to sparse matrix
ch_con_intra_sparse = sp.csr_matrix(ch_con_intra)

# Run cluster test for intra-brain connectivity on fake groups
F_obs_intra, clusters_intra, cluster_p_values_intra, H0_intra, F_obs_plot_intra = stats.statscondCluster(
    data=Alpha_Low,
    freqs_mean=np.arange(7.5, 11),
    ch_con_freq=ch_con_intra_sparse,
    tail=1,
    n_permutations=5000,
    alpha=0.05
)
print("Intra-brain connectivity cluster test completed.")

In [ ]:
cluster_p_values_intra

In [ ]:
viz.viz_2D_topomap_intra(preproc_S1, preproc_S2, F_obs_plot_intra, F_obs_plot_intra, threshold='auto', steps=10, lab=True)

### Cluster Tests for Inter-Brain Connectivity

We now run a cluster-based permutation test for inter-brain connectivity without using connectivity priors.

In [ ]:
F_obs_inter, clusters_inter, cluster_p_values_inter, H0_inter, F_obs_plot_inter = stats.statscondCluster(
    data=data_inter,
    freqs_mean=np.linspace(7.5, 11, data_inter[0].shape[-1]),
    ch_con_freq=None,
    tail=0,
    n_permutations=5000,
    alpha=0.05
)
print("Inter-brain connectivity cluster test completed.")

In [ ]:
cluster_p_values_inter

In [ ]:
viz.viz_2D_topomap_inter(preproc_S1, preproc_S2, F_obs_plot_inter, threshold='auto', steps=10, lab=True)

## 5. Neurobehavioral Correlations

Finally, we simulate a behavioral measure and compute the correlation between the average PSD and the behavior.

In [ ]:
# Simulate a behavioral measure (e.g., cooperation score) for n_samples subjects
behavior = np.random.rand(n_samples)  # values between 0 and 1

# For demonstration, take the average PSD connectivity (from data_sample) across tests
connectivity_mean = np.mean(data_sample.squeeze(), axis=1)

analyses.behav_corr(data=connectivity_mean,
                    behav=behavior,
                    data_name="Connectivity",
                    behav_name="RT",
                    p_thresh=0.05,
                    multiple_corr=True,
                    verbose=True)

## Discussion

- **Univariate Statistics:**  
  We used permutation t-tests (via `statsCond`) to test whether PSD measures significantly differ from 0.

- **Cluster-Based Permutation Testing:**  
  By leveraging spatial adjacency (from sensor positions), cluster-based tests reduce false positives and highlight regions with significant effects.

- **Connectivity Tests:**  
  Fake groups were generated for intra- and inter-brain connectivity to demonstrate cluster testing. Adjust the parameters according to your actual data.

- **Neurobehavioral Correlations:**  
  Linking connectivity measures to behavioral scores helps interpret the functional relevance of INS.

- **EEG vs. fNIRS & Linear vs. Non-Linear:**  
  Although we use EEG data here, remember that fNIRS analysis is often amplitude-based. Moreover, some measures capture linear relationships (e.g., power correlation) while others (e.g., PLV, PLI) capture non-linear dynamics.

Each step should be carefully adjusted and documented for reproducible research.

## Conclusion

In this notebook, we:
- Performed univariate statistical tests on PSD measures.
- Ran cluster-based permutation tests on both intra- and inter-brain connectivity data.
- Explored neurobehavioral correlations linking connectivity to behavior.

These statistical methods are essential to validate the significance of inter-brain synchrony findings and to link neural measures to behavioral performance.